Customer Support Router

In [1]:
pip install -U langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.4/245.4 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 6.5 MB/s eta 0:00:00
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.3.14
    Uninstalling langgraph-sdk-0.3.14:
      Successfully uninstalled langgraph-sdk-0.3.14
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.1
    Uninstalling langgraph-1.2.1:
      Successfully uninstalled langgraph-1.2.1
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.1
    Uninstalling langchain-1.3.1:
      Successfully uninstalled langchain-1.3.1


In [2]:
pip install -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 2.9 MB/s eta 0:00:00


In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata


# LLM Setup
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=userdata.get("apikey")
)


# State Definition
class SupportState(TypedDict):
    query: str
    category: str
    response: str


In [5]:
def classifier_node(state: SupportState):

    prompt = f"""
Classify the customer query into exactly one category:

billing
technical
complaint

Return ONLY the category name.

Customer Query:
{state['query']}
"""

    result = llm.invoke([HumanMessage(content=prompt)])

    return {
        "category": result.content.strip().lower()
    }


In [6]:
def billing_agent(state):
    prompt = f"""
You are a Billing Support Agent.

Customer Query:
{state['query']}

Provide a helpful billing-related response.
"""

    result = llm.invoke([HumanMessage(content=prompt)])

    return {
        "response": result.content
    }

In [7]:
def technical_agent(state):
    prompt = f"""
You are a Technical Support Agent.

Customer Query:
{state['query']}

Provide:
1. Problem diagnosis
2. Possible causes
3. Step-by-step solutions

Keep the response concise.
"""

    result = llm.invoke([HumanMessage(content=prompt)])

    return {
        "response": result.content
    }

In [8]:
def complaint_agent(state):
    prompt = f"""
You are a Complaint Resolution Agent.

Customer Query:
{state['query']}

Acknowledge the complaint, apologize if appropriate,
and suggest next steps.
"""

    result = llm.invoke([HumanMessage(content=prompt)])

    return {
        "response": result.content
    }

In [9]:
def route_query(state: SupportState):

    category = state["category"].strip().lower()

    if "billing" in category:
        return "billing"

    elif "technical" in category:
        return "technical"

    else:
        return "complaint"


In [10]:
workflow = StateGraph(SupportState)

workflow.add_node("classifier", classifier_node)
workflow.add_node("billing", billing_agent)
workflow.add_node("technical", technical_agent)
workflow.add_node("complaint", complaint_agent)

workflow.set_entry_point("classifier")

workflow.add_conditional_edges(
    "classifier",
    route_query,
    {
        "billing": "billing",
        "technical": "technical",
        "complaint": "complaint"
    }
)

workflow.add_edge("billing", END)
workflow.add_edge("technical", END)
workflow.add_edge("complaint", END)

app = workflow.compile()


In [12]:
query = input("Enter Customer Query: ")

result = app.invoke({
    "query": query,
    "category": "",
    "response": ""
})

print("\nCategory:", result["category"])
print("\nFinal Response:\n")
print(result["response"])

Enter Customer Query: app is crasing

Category: technical

Final Response:

Here's a diagnosis and solutions for your crashing app:

**1. Problem Diagnosis:**
The application is terminating unexpectedly, preventing normal usage.

**2. Possible Causes:**
*   Outdated app or operating system (OS).
*   Corrupted app data or cache.
*   Insufficient device memory or storage.
*   Software bugs within the app itself.
*   Conflicts with other installed applications.

**3. Step-by-Step Solutions:**

1.  **Restart App:** Close the app completely and reopen it.
2.  **Restart Device:** Turn your phone/tablet off and then back on.
3.  **Clear App Cache/Data:** Go to device Settings > Apps > [Crashing App] > Storage > Clear Cache, then Clear Data (note: Clear Data will erase app settings/logins).
4.  **Update App & OS:** Check your app store for updates for the app, and your device settings for OS updates.
5.  **Reinstall App:** Uninstall the app, then download and install it again from the app stor